# Cost comparison

Report metrics from SQLite. The cell below sets the input and output token prices.

In [ ]:
# Model rate/tariff in USD per million tokens (input, output, rate date).
rates = {
    ("openai", "chat_completions", "gpt-6-luna"): ("0.10", "0.50", "2026-11-13"),
    ("openai", "decisions", "gpt-6-luna"): ("0.09414", "0", "2026-11-11"),
    ("openai", "chat_completions", "gpt-5-nano"): ("0.05", "0.40", "2026-11-05"),
    ("gemini", "chat_completions", "gemini-flash-lite-latest"): ("0.25", "1.50", "2026-11-06"),
}

## Cost by model

Table 1 includes saved games on SQLite:
- **Total price** sums each model API request’s `(input tokens × input rate + output tokens × output rate) ÷ 1M`.
- **Cost $/M** uses the actual combined token count: `total price ÷ (input tokens + output tokens) × 1M`.
- **Per run** divides total price by the number of completed runs for each model.

In [ ]:
from collections import defaultdict
from decimal import Decimal
import json
import sys

import pandas as pd

from data_paths import DATA_DIR, METRICS_PATH, REPO_ROOT

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from agent.metrics import database
from agent.metrics import eligible_games

db_path = METRICS_PATH
million = Decimal(1_000_000)

def token_cost(row, tariff):
    inputs, outputs = row["input_tokens"], row["output_tokens"]
    if inputs is None or outputs is None or tariff is None:
        return None
    return (inputs * Decimal(tariff[0]) + outputs * Decimal(tariff[1])) / million

query = """
    SELECT q.input_tokens, q.output_tokens FROM model_calls q WHERE q.run_id=?
"""
by_selector = defaultdict(list)
games_by_selector = defaultdict(list)
by_trace = {}
tariffs = {}
with database(db_path, readonly=True) as db:
    db.execute("BEGIN")
    for run in eligible_games(db):
        trace_id = run["id"]
        selector = (run["profile"] or "Unknown", run["model_id"] or "Unknown")
        requests = db.execute(query, (trace_id,)).fetchall()
        games_by_selector[selector].append(trace_id)
        rows = []
        route = (run["gw_provider"], run["api"], run["model_id"])
        tariff = rates.get(route) if run["gateway"] == "direct" else None
        if tariff is not None:
            tariffs[(*selector, *tariff)] = None
        for record in requests:
            rows.append(dict(record, cost=token_cost(record, tariff)))
        by_trace[trace_id] = (selector, rows, run["saved_at"], run["decisions"])
        by_selector[selector].extend(rows)

print(f"{len(by_trace)} eligible SQLite games.")

In [ ]:
def complete_sum(rows, column):
    if rows and all(r[column] is not None for r in rows):
        return sum(r[column] for r in rows)

def summarize(rows):
    inputs = complete_sum(rows, "input_tokens")
    outputs = complete_sum(rows, "output_tokens")
    total = complete_sum(rows, "cost")
    rate = None
    if total is not None and inputs is not None and outputs is not None and inputs + outputs > 0:
        rate = million * total / (inputs + outputs)
    return {"Input tokens": inputs, "Output tokens": outputs, "Total price": total, "Cost $/M": rate}

def show(rows):
    frame = pd.DataFrame(rows)
    if "Model" in frame and "Profile" in frame:
        frame["Model"] = [
            f"{profile}:{model}" if profile and model else model
            for profile, model in zip(frame["Profile"], frame["Model"])
        ]
    frame = frame.drop(columns=["Profile", "API"], errors="ignore")
    for col in ("Per run", "Total price", "Cost $/M", "Input $/M", "Output $/M"):
        if col in frame:
            frame[col] = frame[col].map(lambda v: f"${Decimal(v):.5f}" if pd.notna(v) else "Unknown")
    display(frame.fillna("Unknown").style.hide(axis="index"))

selector_columns = ("Profile", "Model")
overview = [dict(zip(selector_columns, key), **summarize(rows)) for key, rows in by_selector.items()]
for summary in overview:
    selector = tuple(summary[col] for col in selector_columns)
    games_played = len(games_by_selector[selector])
    summary["Runs"] = games_played
    total = summary["Total price"]
    summary["Per run"] = total / games_played if total is not None else None
if overview:
    print("Table 1: Token usage and spend by model")
    show(overview)

    if tariffs:
        rate_columns = (*selector_columns, "Input $/M", "Output $/M", "Rate date")
        print("Table 2: Input and output token rates")
        show([dict(zip(rate_columns, key)) for key in tariffs])
else:
    print("No matching runs.")

## Cost by game

In [ ]:
trace_path = DATA_DIR / "agent-traces.json"
trace_ids = set(json.loads(trace_path.read_text())["runs"]) if trace_path.exists() else set()
by_trace = {rid: row for rid, row in by_trace.items() if rid in trace_ids}
run_rows = []
if len({key[:8] for key in by_trace}) != len(by_trace):
    raise ValueError("Trace IDs share the same eight-character prefix")
for trace_id, (selector, rows, ended, decision_count) in by_trace.items():
    end_time = pd.to_datetime(ended, utc=True).tz_convert("America/Los_Angeles").strftime("%Y-%m-%d %H:%M %Z")
    run_rows.append({"Trace ID": trace_id[:8], "Saved at": end_time, "# decisions": decision_count,
                     "Profile": selector[0], "Model": selector[1], **summarize(rows)})
if run_rows:
    print("Table 3: Estimated cost of games on trace store")
    show(run_rows)